# **1,000-Cluster Solar Allocation Test**

This notebook tests a possible pre-allocation approach for solar resource clustering.

The goal is to start with a fixed cluster budget, here **1,000 total clusters**, and allocate those clusters across counties or county-region pairs before running PowerGenome.

The logic is:

1. Give every valid county or county-region group at least **1 baseline cluster**.
2. Calculate how much variation exists within each group using **LCOE standard deviation**.
3. Allocate the remaining clusters proportionally to groups with more LCOE variation.
4. Cap each group by the number of candidate solar rows available, so we do not ask PowerGenome to create more clusters than there are candidate sites.

This is not yet modifying PowerGenome directly. It creates an allocation table that can be used as the basis for a future resource-clustering setup.

## Step 1: Set file paths

This section points the notebook to the local files used in the WECC county-solar test.

The important inputs are:

- The **county-enriched solar metadata file**, which contains the candidate solar rows and county assignments.
- The **WECC model definition**, which maps the lower-level `p` regions into the 34 WECC model regions.
- The output folder where this notebook saves the allocation results.

The paths are local to my machine, so they may need to be updated if this notebook is run somewhere else.

In [172]:
from pathlib import Path
import pandas as pd
import numpy as np
import yaml

PROJECT_ROOT = Path("/Users/laurenvo/Documents/Github/solar-county-analysis")
SWITCH_REPO = Path("/Users/laurenvo/Documents/Switch-USA-PG-ReEDS")

SOLAR_META_PATH = SWITCH_REPO / "pg/extra_inputs/resource_groups_wecc_county_solar_test/ReEDS-cpas-patched/solar_lcoe_ReEDS_pg_schema_with_county_group.csv"

MODEL_DEF_PATH = SWITCH_REPO / "pg/settings_wecc_county_solar_5clusters/model_definition.yml"

OUT_DIR = PROJECT_ROOT / "clustering_county" / "outputs" / "allocation_1000"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("SOLAR_META_PATH exists:", SOLAR_META_PATH.exists(), SOLAR_META_PATH)
print("MODEL_DEF_PATH exists:", MODEL_DEF_PATH.exists(), MODEL_DEF_PATH)
print("OUT_DIR:", OUT_DIR)

SOLAR_META_PATH exists: True /Users/laurenvo/Documents/Switch-USA-PG-ReEDS/pg/extra_inputs/resource_groups_wecc_county_solar_test/ReEDS-cpas-patched/solar_lcoe_ReEDS_pg_schema_with_county_group.csv
MODEL_DEF_PATH exists: True /Users/laurenvo/Documents/Switch-USA-PG-ReEDS/pg/settings_wecc_county_solar_5clusters/model_definition.yml
OUT_DIR: /Users/laurenvo/Documents/Github/solar-county-analysis/clustering_county/outputs/allocation_1000


In [173]:
list((SWITCH_REPO / "pg").glob("settings_wecc*/model_definition.yml"))

[PosixPath('/Users/laurenvo/Documents/Switch-USA-PG-ReEDS/pg/settings_wecc_county_solar_test/model_definition.yml'),
 PosixPath('/Users/laurenvo/Documents/Switch-USA-PG-ReEDS/pg/settings_wecc_county_test/model_definition.yml'),
 PosixPath('/Users/laurenvo/Documents/Switch-USA-PG-ReEDS/pg/settings_wecc_baseline_validated/model_definition.yml'),
 PosixPath('/Users/laurenvo/Documents/Switch-USA-PG-ReEDS/pg/settings_wecc_county_solar_5clusters/model_definition.yml')]

## Step 2: Load WECC region definitions and solar metadata

This section loads the model-region mapping and the county-enriched solar metadata.

The model definition maps PowerGenome/ReEDS `p` regions into the 34 WECC model regions used in the working county-solar test.

The solar metadata contains candidate utility-scale solar rows, including:

- `ipm_region`: the original region label for each candidate site
- `county_group`: the county FIPS assignment
- `lcoe`: levelized cost of energy
- `cf`: capacity factor
- `capacity_mw`: candidate resource capacity

The notebook uses these fields to decide how many clusters each county or county-region group should receive.

In [174]:
with open(MODEL_DEF_PATH, "r") as f:
    model_def = yaml.safe_load(f)

region_aggregations = model_def["region_aggregations"]

# p-region -> WECC model region
p_to_model_region = {}
for model_region, p_regions in region_aggregations.items():
    for p in p_regions:
        p_to_model_region[str(p)] = model_region

print("Number of WECC model regions:", len(region_aggregations))
print("Number of p-regions mapped:", len(p_to_model_region))

solar = pd.read_csv(SOLAR_META_PATH)

print("Solar rows:", len(solar))
print("Columns:")
print(solar.columns.tolist())

solar.head()

Number of WECC model regions: 34
Number of p-regions mapped: 34
Solar rows: 405737
Columns:
['Area', 'd_trans', 'd_sub', 'd_road', 'd_load_750', 'd_existing', 'd_plannedF', 'm_slope', 'm_popden', 'm_HMI', 'm_primeFarmland', 'incap', 'CPA_ID', 'm_aspect', 'm_aspect_min', 'Shape_Leng', 'm_landcover', 'exFacil', 'plFacil', 'Qual_Coal', 'Qual_Emp', 'Qual_Brown', 'anyQual', 'Qual_noBF', 'SocialImpa', 'EnviroImpa', 'Shape_Le_1', 'Shape_Area', 'pop_density_bin', 'tech', 'metro_id', 'metro_region', 'cpa_mw', 'cf', 'path', 'resource_annuity', 'resource_fom', 'interconnect_annuity', 'lcoe', 'interconnect_capex_mw', 'total_interconnect_km', 'offshore_interconnect_km', 'ipm_region', 'county_group', 'capacity_mw']


/var/folders/dg/bllrnyzj2fzf93_myxywvnnm0000gn/T/ipykernel_63441/1479324811.py:15: DtypeWarning: Columns (30) have mixed types. Specify dtype option on import or set low_memory=False.
  solar = pd.read_csv(SOLAR_META_PATH)


,Area,d_trans,d_sub,d_road,d_load_750,d_existing,d_plannedF,m_slope,m_popden,m_HMI,...,resource_annuity,resource_fom,interconnect_annuity,lcoe,interconnect_capex_mw,total_interconnect_km,offshore_interconnect_km,ipm_region,county_group,capacity_mw
0,2.00,19.142255,26.470939,0.000000,158.648693,265.038776,246.071077,0.125000,17.375000,0.780994,...,48359.91,15221.571,23691.982,45.182354,497798.70,143.821594,0.0,p1,53073.0,4.500000
1,3.75,13.030586,20.955390,1.085683,156.003598,264.714080,245.267324,1.600000,34.266667,0.786600,...,48359.91,15221.571,23711.625,44.400440,498211.40,140.985809,0.0,p1,53073.0,4.218750
2,3.00,35.946548,45.514552,6.423029,156.322476,252.099160,235.024352,9.833333,0.000000,0.097131,...,48359.91,15221.571,25068.760,45.527447,526726.56,150.642654,0.0,p1,53073.0,26.999998
3,2.00,22.097498,30.668918,0.000000,149.529784,252.050029,233.665701,3.375000,35.125000,0.544276,...,48359.91,15221.571,22436.877,43.882450,471427.34,135.036041,0.0,p1,53073.0,2.250000
4,2.25,18.546952,28.017060,0.126662,145.881474,249.425370,230.785601,8.777778,1.888889,0.429502,...,48359.91,15221.571,21848.154,43.582115,459057.50,130.457626,0.0,p1,53073.0,20.250000


## Step 3: Filter to active WECC solar rows

This section keeps only the solar rows that belong to the active WECC model regions.

It also maps each lower-level `ipm_region` into a WECC `model_region`.

After this step, the key validation numbers should be checked:

- Number of WECC model regions
- Number of unique counties
- Number of unique county-region pairs

The county-region count is expected to be larger than the unique-county count because the same county can appear in more than one model region. For the current WECC setup, this is why we distinguish between:

- **unique counties**
- **unique `(model_region, county_group)` pairs**

The county-region version is closer to how the current PowerGenome clustering test was run.

In [175]:
region_col = "ipm_region"
county_col = "county_group"
metric_col = "lcoe"   # use LCOE for first version

required = [region_col, county_col, metric_col]
missing = [c for c in required if c not in solar.columns]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

wecc = solar.copy()
wecc[region_col] = wecc[region_col].astype(str)

# Keep only p-regions included in the WECC model definition
wecc = wecc[wecc[region_col].isin(p_to_model_region.keys())].copy()

# Add WECC model region
wecc["model_region"] = wecc[region_col].map(p_to_model_region)

# Keep valid county + metric rows
wecc = wecc.dropna(subset=["model_region", county_col, metric_col]).copy()
wecc[county_col] = (
    pd.to_numeric(wecc[county_col], errors="coerce")
    .astype("Int64")
    .astype(str)
    .str.zfill(5)
)

print("Filtered WECC solar rows:", len(wecc))
print("Unique model regions:", wecc["model_region"].nunique())
print("Unique counties:", wecc[county_col].nunique())
print("Unique county-region pairs:", wecc[["model_region", county_col]].drop_duplicates().shape[0])

wecc.head()

Filtered WECC solar rows: 118233
Unique model regions: 34
Unique counties: 419
Unique county-region pairs: 739


,Area,d_trans,d_sub,d_road,d_load_750,d_existing,d_plannedF,m_slope,m_popden,m_HMI,...,resource_fom,interconnect_annuity,lcoe,interconnect_capex_mw,total_interconnect_km,offshore_interconnect_km,ipm_region,county_group,capacity_mw,model_region
0,2.00,19.142255,26.470939,0.000000,158.648693,265.038776,246.071077,0.125000,17.375000,0.780994,...,15221.571,23691.982,45.182354,497798.70,143.821594,0.0,p1,53073,4.500000,WA1
1,3.75,13.030586,20.955390,1.085683,156.003598,264.714080,245.267324,1.600000,34.266667,0.786600,...,15221.571,23711.625,44.400440,498211.40,140.985809,0.0,p1,53073,4.218750,WA1
2,3.00,35.946548,45.514552,6.423029,156.322476,252.099160,235.024352,9.833333,0.000000,0.097131,...,15221.571,25068.760,45.527447,526726.56,150.642654,0.0,p1,53073,26.999998,WA1
3,2.00,22.097498,30.668918,0.000000,149.529784,252.050029,233.665701,3.375000,35.125000,0.544276,...,15221.571,22436.877,43.882450,471427.34,135.036041,0.0,p1,53073,2.250000,WA1
4,2.25,18.546952,28.017060,0.126662,145.881474,249.425370,230.785601,8.777778,1.888889,0.429502,...,15221.571,21848.154,43.582115,459057.50,130.457626,0.0,p1,53073,20.250000,WA1


## Step 4: Define the cluster allocation function

This function assigns a fixed number of clusters across groups.

The function works in three stages:

1. **Baseline allocation**  
   Every valid group gets 1 cluster first.

2. **Remaining budget allocation**  
   The remaining cluster budget is distributed based on each group's LCOE standard deviation. Groups with more variation in LCOE receive more extra clusters.

3. **Candidate-row cap**  
   A group cannot receive more clusters than the number of candidate solar rows available in that group.

For example, if a county-region group only has 3 candidate solar rows, it cannot receive more than 3 clusters, even if the allocation formula would otherwise assign more.

In [176]:
def allocate_cluster_budget(group_stats, total_pool=1000):
    out = group_stats.copy()

    if total_pool < len(out):
        raise ValueError(
            f"total_pool={total_pool} is smaller than number of groups={len(out)}. "
            "Cannot give every group at least 1 cluster."
        )

    # Every group starts with 1 cluster
    out["assigned_clusters"] = 1

    # Cannot assign more clusters than candidate rows
    out["max_possible_clusters"] = out["candidate_rows"]
    out["remaining_capacity"] = out["max_possible_clusters"] - out["assigned_clusters"]
    out["remaining_capacity"] = out["remaining_capacity"].clip(lower=0)

    remaining_budget = total_pool - out["assigned_clusters"].sum()

    # Use metric std as weight. If std is zero everywhere, fallback to candidate row count.
    out["weight"] = out["metric_std"].fillna(0).clip(lower=0)

    if out["weight"].sum() == 0:
        out["weight"] = out["candidate_rows"]

    while remaining_budget > 0:
        eligible = out["remaining_capacity"] > 0

        if not eligible.any():
            print("No remaining candidate capacity. Could not use full budget.")
            break

        weights = out.loc[eligible, "weight"].copy()

        if weights.sum() == 0:
            weights[:] = 1

        raw_add = remaining_budget * weights / weights.sum()
        add_floor = np.floor(raw_add).astype(int)

        # Cap by remaining candidate capacity
        add_floor = np.minimum(add_floor, out.loc[eligible, "remaining_capacity"].astype(int))

        if add_floor.sum() > 0:
            out.loc[eligible, "assigned_clusters"] += add_floor
            out.loc[eligible, "remaining_capacity"] -= add_floor
            remaining_budget -= int(add_floor.sum())
        else:
            # Allocate one-by-one by largest fractional remainder
            remainders = (raw_add - np.floor(raw_add)).sort_values(ascending=False)

            made_progress = False
            for idx in remainders.index:
                if remaining_budget <= 0:
                    break
                if out.loc[idx, "remaining_capacity"] > 0:
                    out.loc[idx, "assigned_clusters"] += 1
                    out.loc[idx, "remaining_capacity"] -= 1
                    remaining_budget -= 1
                    made_progress = True

            if not made_progress:
                break

    out = out.drop(columns=["weight"])

    return out

## Step 5A: County-region allocation

This version groups by:

`model_region + county_group`

This is the version that most closely matches the current PowerGenome WECC county-solar setup, because the previous clustering test created UtilityPV resources at the county-region level.

How to read the output:

- `Groups`: number of valid county-region pairs
- `Assigned total`: total number of clusters allocated
- `Groups with at least 1`: confirms whether every county-region pair received a baseline cluster
- `Groups with more than 1`: shows how many groups received extra clusters beyond the baseline
- `Max assigned`: largest number of clusters assigned to any one county-region pair

If this section shows `Assigned total = 1000` and `Groups with at least 1 = Groups`, then the allocation is working as intended for the county-region setup.

In [177]:
TOTAL_POOL = 1000

county_region_stats = (
    wecc.groupby(["model_region", county_col])
    .agg(
        candidate_rows=(metric_col, "size"),
        metric_mean=(metric_col, "mean"),
        metric_std=(metric_col, "std"),
        metric_min=(metric_col, "min"),
        metric_max=(metric_col, "max"),
        capacity_mw=("capacity_mw", "sum") if "capacity_mw" in wecc.columns else (metric_col, "size"),
    )
    .reset_index()
)

county_region_stats["metric_std"] = county_region_stats["metric_std"].fillna(0)

county_region_alloc = allocate_cluster_budget(county_region_stats, TOTAL_POOL)

print("===== COUNTY-REGION ALLOCATION =====")
print("Groups:", len(county_region_alloc))
print("Assigned total:", county_region_alloc["assigned_clusters"].sum())
print("Groups with at least 1:", (county_region_alloc["assigned_clusters"] >= 1).sum())
print("Groups with more than 1:", (county_region_alloc["assigned_clusters"] > 1).sum())
print("Max assigned:", county_region_alloc["assigned_clusters"].max())

county_region_alloc.sort_values("assigned_clusters", ascending=False).head(20)

===== COUNTY-REGION ALLOCATION =====
Groups: 739
Assigned total: 1000
Groups with at least 1: 739
Groups with more than 1: 133
Max assigned: 24


,model_region,county_group,candidate_rows,metric_mean,metric_std,metric_min,metric_max,capacity_mw,assigned_clusters,max_possible_clusters,remaining_capacity
143,CO1,08077,24,30.401714,8.203661,24.375786,42.098698,1096.122216,24,24,0
571,UT2,49047,47,37.588225,6.247358,24.492857,41.278830,2649.342724,16,47,31
73,CA2,06107,146,35.047809,5.884674,25.838688,40.510708,8403.504564,12,146,134
65,CA2,06029,260,35.797041,5.790704,24.778572,41.033405,23059.386363,11,260,249
685,WY1,56035,522,34.063224,5.342305,28.571276,46.608837,59490.380793,9,522,513
581,WA1,53035,42,37.359033,5.423266,31.584350,47.017520,1033.045325,9,42,33
557,UT1,49055,114,37.676476,5.216206,30.923578,48.031300,7404.401079,8,114,106
590,WA1,53073,92,39.930052,5.292319,31.383036,45.734040,2452.372229,8,92,84
109,CA4,06083,102,39.412984,5.082561,25.280487,42.797062,6645.966188,7,102,95
14,AZ2,04005,225,37.879707,4.921276,23.610842,42.467392,17330.192205,7,225,218


## Step 5B: County-only allocation

This version groups by:

`county_group`

This is closer to the conceptual example where every unique county receives 1 baseline cluster first.

How to read the output:

- `Counties`: number of unique counties with valid solar candidate rows
- `Assigned total`: total number of clusters allocated
- `Counties with at least 1`: confirms every county received a baseline cluster
- `Counties with more than 1`: shows how many counties received extra clusters
- `Max assigned`: largest number of clusters assigned to any one county

This version is useful for thinking about county-level equity/coverage. However, if a county appears in multiple model regions, an additional step would be needed to split that county's assigned clusters back across model regions before using it directly in PowerGenome.

In [178]:
county_stats = (
    wecc.groupby([county_col])
    .agg(
        candidate_rows=(metric_col, "size"),
        metric_mean=(metric_col, "mean"),
        metric_std=(metric_col, "std"),
        metric_min=(metric_col, "min"),
        metric_max=(metric_col, "max"),
        capacity_mw=("capacity_mw", "sum") if "capacity_mw" in wecc.columns else (metric_col, "size"),
        model_regions=("model_region", lambda x: ",".join(sorted(set(x)))),
        n_model_regions=("model_region", "nunique"),
    )
    .reset_index()
)

county_stats["metric_std"] = county_stats["metric_std"].fillna(0)

county_alloc = allocate_cluster_budget(county_stats, TOTAL_POOL)

print("===== COUNTY-ONLY ALLOCATION =====")
print("Counties:", len(county_alloc))
print("Assigned total:", county_alloc["assigned_clusters"].sum())
print("Counties with at least 1:", (county_alloc["assigned_clusters"] >= 1).sum())
print("Counties with more than 1:", (county_alloc["assigned_clusters"] > 1).sum())
print("Max assigned:", county_alloc["assigned_clusters"].max())

county_alloc.sort_values("assigned_clusters", ascending=False).head(20)

===== COUNTY-ONLY ALLOCATION =====
Counties: 419
Assigned total: 1000
Counties with at least 1: 419
Counties with more than 1: 230
Max assigned: 63


,county_group,candidate_rows,metric_mean,metric_std,metric_min,metric_max,capacity_mw,model_regions,n_model_regions,assigned_clusters,max_possible_clusters,remaining_capacity
361,53009,148,48.173822,8.623118,30.765574,55.759003,9788.873481,"WA1,WA2",2,63,148,85
374,53035,42,37.359033,5.423266,31.584350,47.017520,1033.045325,WA1,1,13,42,29
413,56035,522,34.063224,5.342305,28.571276,46.608837,59490.380793,WY1,1,12,522,510
393,53073,92,39.930052,5.292319,31.383036,45.734040,2452.372229,WA1,1,12,92,80
346,49037,760,36.377138,5.222725,30.175661,47.407833,62711.172181,"AZ1,AZ2,AZ3,NM1,UT1,WY1",6,11,760,749
350,49045,489,32.444977,4.693385,27.363312,43.479042,41053.506440,"ID1,NV2,UT1",3,9,489,480
336,49017,221,41.539199,4.753059,33.880054,49.197147,13101.195113,"AZ2,NV2,UT1,WY1",4,9,221,212
339,49023,462,33.235236,4.778730,27.393309,43.989872,45215.861831,"NV2,UT1",2,9,462,453
26,06023,123,47.958427,4.611376,38.397150,60.967754,5511.561918,"CA1,CA4,OR1",3,9,123,114
66,06105,49,36.544313,4.644642,30.754414,46.615124,1978.584514,"CA1,CA4",2,9,49,40


### Important distinction: county vs. county-region

A county and a county-region pair are not the same thing.

A single county can appear in more than one WECC model region. Because of that:

- The **county-only** count is the number of unique county FIPS codes.
- The **county-region** count is the number of unique `(model_region, county_group)` combinations.

The county-region count is larger because it treats the same county in two different model regions as two separate clustering groups.

## Check LCOE and CF variability for large counties

This section looks at counties with many candidate solar rows, starting with county group `04001`.

For each county, we check:

- number of candidate rows
- LCOE mean, standard deviation, min, max
- CF mean, standard deviation, min, max
- whether CF still varies a lot within LCOE-based bins

If CF variation remains high within LCOE-based bins, then LCOE-only clustering may not be enough and we may need to cluster on both LCOE and CF.

In [179]:
# Identify LCOE and CF columns
print("Available columns:")
print(wecc.columns.tolist())

lcoe_col = "lcoe"

# Try to detect CF column
possible_cf_cols = ["cf", "capacity_factor", "mean_cf", "profile_cf"]
cf_col = None

for c in possible_cf_cols:
    if c in wecc.columns:
        cf_col = c
        break

if cf_col is None:
    raise ValueError("Could not find CF column. Check the printed columns above.")

print("Using LCOE column:", lcoe_col)
print("Using CF column:", cf_col)

Available columns:
['Area', 'd_trans', 'd_sub', 'd_road', 'd_load_750', 'd_existing', 'd_plannedF', 'm_slope', 'm_popden', 'm_HMI', 'm_primeFarmland', 'incap', 'CPA_ID', 'm_aspect', 'm_aspect_min', 'Shape_Leng', 'm_landcover', 'exFacil', 'plFacil', 'Qual_Coal', 'Qual_Emp', 'Qual_Brown', 'anyQual', 'Qual_noBF', 'SocialImpa', 'EnviroImpa', 'Shape_Le_1', 'Shape_Area', 'pop_density_bin', 'tech', 'metro_id', 'metro_region', 'cpa_mw', 'cf', 'path', 'resource_annuity', 'resource_fom', 'interconnect_annuity', 'lcoe', 'interconnect_capex_mw', 'total_interconnect_km', 'offshore_interconnect_km', 'ipm_region', 'county_group', 'capacity_mw', 'model_region']
Using LCOE column: lcoe
Using CF column: cf


In [180]:
target_county = "04001"

county_df = wecc[wecc[county_col] == target_county].copy()

print("County:", target_county)
print("Candidate rows:", len(county_df))

summary_04001 = county_df[[lcoe_col, cf_col]].describe().T
summary_04001["range"] = summary_04001["max"] - summary_04001["min"]

summary_04001

County: 04001
Candidate rows: 1720


,count,mean,std,min,25%,50%,75%,max,range
lcoe,1720.0,27.885293,2.223027,23.177225,26.319950,27.585657,29.311698,33.920410,10.743185
cf,1720.0,0.309503,0.004098,0.296853,0.307586,0.310752,0.312444,0.315813,0.018960


In [181]:
# Create LCOE quantile bins for this county.
# This approximates asking: if we group by LCOE, how much CF variation remains inside each LCOE group?

n_bins = 5

county_df = county_df.dropna(subset=[lcoe_col, cf_col]).copy()

county_df["lcoe_bin"] = pd.qcut(
    county_df[lcoe_col],
    q=min(n_bins, county_df[lcoe_col].nunique()),
    duplicates="drop"
)

lcoe_bin_summary = (
    county_df.groupby("lcoe_bin")
    .agg(
        candidate_rows=(lcoe_col, "size"),
        lcoe_mean=(lcoe_col, "mean"),
        lcoe_std=(lcoe_col, "std"),
        lcoe_min=(lcoe_col, "min"),
        lcoe_max=(lcoe_col, "max"),
        cf_mean=(cf_col, "mean"),
        cf_std=(cf_col, "std"),
        cf_min=(cf_col, "min"),
        cf_max=(cf_col, "max"),
    )
    .reset_index()
)

lcoe_bin_summary["lcoe_range"] = lcoe_bin_summary["lcoe_max"] - lcoe_bin_summary["lcoe_min"]
lcoe_bin_summary["cf_range"] = lcoe_bin_summary["cf_max"] - lcoe_bin_summary["cf_min"]

lcoe_bin_summary

,lcoe_bin,candidate_rows,lcoe_mean,lcoe_std,lcoe_min,lcoe_max,cf_mean,cf_std,cf_min,cf_max,lcoe_range,cf_range
0,"(23.176, 25.989]",344,25.084785,0.685944,23.177225,25.985270,0.311490,0.002094,0.307066,0.315098,2.808045,0.008033
1,"(25.989, 27.092]",344,26.549940,0.304089,25.990318,27.089703,0.310993,0.002606,0.305327,0.314866,1.099385,0.009539
2,"(27.092, 28.211]",344,27.607433,0.315982,27.093151,28.206072,0.310339,0.003166,0.303893,0.315813,1.112921,0.011920
3,"(28.211, 29.685]",344,28.962989,0.416925,28.217978,29.680035,0.308303,0.004469,0.298843,0.315813,1.462057,0.016971
4,"(29.685, 33.92]",344,31.221318,1.368352,29.705307,33.920410,0.306390,0.004995,0.296853,0.315668,4.215103,0.018815


In [182]:
# Look at the largest counties by candidate row count

large_counties = (
    wecc.groupby(county_col)
    .size()
    .reset_index(name="candidate_rows")
    .sort_values("candidate_rows", ascending=False)
    .head(10)
)

large_counties

,county_group,candidate_rows
231,32007,2132
2,04005,2078
239,32023,1841
0,04001,1720
414,56037,1647
9,04017,1556
8,04015,1447
292,41045,1403
282,41025,1279
399,56007,1167


In [183]:
large_county_ids = large_counties[county_col].tolist()

large_county_variability = (
    wecc[wecc[county_col].isin(large_county_ids)]
    .groupby(county_col)
    .agg(
        candidate_rows=(lcoe_col, "size"),
        lcoe_mean=(lcoe_col, "mean"),
        lcoe_std=(lcoe_col, "std"),
        lcoe_min=(lcoe_col, "min"),
        lcoe_max=(lcoe_col, "max"),
        cf_mean=(cf_col, "mean"),
        cf_std=(cf_col, "std"),
        cf_min=(cf_col, "min"),
        cf_max=(cf_col, "max"),
    )
    .reset_index()
)

large_county_variability["lcoe_range"] = large_county_variability["lcoe_max"] - large_county_variability["lcoe_min"]
large_county_variability["cf_range"] = large_county_variability["cf_max"] - large_county_variability["cf_min"]

large_county_variability.sort_values("candidate_rows", ascending=False)

,county_group,candidate_rows,lcoe_mean,lcoe_std,lcoe_min,lcoe_max,cf_mean,cf_std,cf_min,cf_max,lcoe_range,cf_range
4,32007,2132,36.396738,2.009973,31.445559,43.308426,0.295105,0.004333,0.281558,0.309496,11.862867,0.027938
1,04005,2078,31.450453,3.062578,23.610842,42.467392,0.308489,0.004185,0.293749,0.317243,18.856550,0.023494
5,32023,1841,33.538328,3.891605,23.362118,41.945328,0.314338,0.006673,0.296283,0.329443,18.583210,0.033159
0,04001,1720,27.885293,2.223027,23.177225,33.920410,0.309503,0.004098,0.296853,0.315813,10.743185,0.018960
9,56037,1647,28.913623,2.656980,25.057420,42.561260,0.292128,0.003152,0.283361,0.300811,17.503840,0.017450
3,04017,1556,29.858732,1.771087,26.538240,36.259624,0.309509,0.003603,0.299446,0.316826,9.721384,0.017380
2,04015,1447,28.002804,4.466206,22.663147,39.776836,0.312223,0.004325,0.300358,0.322462,17.113689,0.022104
7,41045,1403,32.426957,4.374608,26.806355,44.730553,0.289056,0.005488,0.273930,0.302586,17.924198,0.028656
6,41025,1279,36.100379,2.997894,30.727543,43.814934,0.292585,0.004047,0.282265,0.306178,13.087391,0.023913
8,56007,1167,33.274466,2.176215,30.025312,42.037315,0.291485,0.003085,0.283211,0.298453,12.012003,0.015242


## Add county names to output tables

This section adds readable county names to the allocation outputs.

The allocation tables already include county FIPS codes through `county_group`. Adding county names makes the CSVs easier to review.

In [184]:
# Load county FIPS/name lookup from Census reference file
county_lookup_url = "https://www2.census.gov/geo/docs/reference/codes/files/national_county.txt"

county_lookup = pd.read_csv(
    county_lookup_url,
    header=None,
    names=["state_abbrev", "state_fips", "county_fips", "county_name", "class_fips"],
    dtype={"state_fips": str, "county_fips": str}
)

county_lookup["county_group"] = county_lookup["state_fips"].str.zfill(2) + county_lookup["county_fips"].str.zfill(3)
county_lookup["county_name_full"] = county_lookup["county_name"] + ", " + county_lookup["state_abbrev"]

county_lookup = county_lookup[["county_group", "county_name_full"]].drop_duplicates()

county_lookup.head()

,county_group,county_name_full
0,01001,"Autauga County, AL"
1,01003,"Baldwin County, AL"
2,01005,"Barbour County, AL"
3,01007,"Bibb County, AL"
4,01009,"Blount County, AL"


In [185]:
# Add county names to allocation outputs
county_region_alloc = county_region_alloc.merge(
    county_lookup,
    left_on=county_col,
    right_on="county_group",
    how="left"
)

county_alloc = county_alloc.merge(
    county_lookup,
    left_on=county_col,
    right_on="county_group",
    how="left"
)

# Clean duplicate helper column if needed
if "county_group_y" in county_region_alloc.columns:
    county_region_alloc = county_region_alloc.rename(columns={"county_group_x": "county_group"})
    county_region_alloc = county_region_alloc.drop(columns=["county_group_y"])

if "county_group_y" in county_alloc.columns:
    county_alloc = county_alloc.rename(columns={"county_group_x": "county_group"})
    county_alloc = county_alloc.drop(columns=["county_group_y"])

county_region_alloc.head()

,model_region,county_group,candidate_rows,metric_mean,metric_std,metric_min,metric_max,capacity_mw,assigned_clusters,max_possible_clusters,remaining_capacity,county_name_full
0,AZ1,04005,1214,30.351884,1.394478,26.963272,35.567307,143152.699444,1,1214,1213,"Coconino County, AZ"
1,AZ1,04012,377,26.830686,0.928475,23.984173,31.331976,36432.433853,1,377,376,"La Paz County, AZ"
2,AZ1,04013,51,29.281596,1.094344,27.556498,31.534460,4170.152205,1,51,50,"Maricopa County, AZ"
3,AZ1,04015,981,25.356120,1.456630,22.663147,29.629540,87845.157974,1,981,980,"Mohave County, AZ"
4,AZ1,04025,497,28.233114,1.359513,25.640427,31.274826,46211.229409,1,497,496,"Yavapai County, AZ"


## Step 5C: Composite CF + LCOE + capacity-weighted allocation

In [186]:
TOTAL_POOL = 1000
W_CF = 0.5
W_LCOE = 0.5

cap_col = "capacity_mw"

if cap_col not in wecc.columns:
    raise ValueError(f"Missing capacity column: {cap_col}")

county_region_composite_stats = (
    wecc.groupby(["model_region", county_col])
    .agg(
        candidate_rows=(lcoe_col, "size"),
        total_capacity_mw=(cap_col, "sum"),
        cf_std=(cf_col, "std"),
        cf_min=(cf_col, "min"),
        cf_max=(cf_col, "max"),
        lcoe_std=(lcoe_col, "std"),
        lcoe_min=(lcoe_col, "min"),
        lcoe_max=(lcoe_col, "max"),
    )
    .reset_index()
)

county_region_composite_stats["cf_std"] = county_region_composite_stats["cf_std"].fillna(0)
county_region_composite_stats["lcoe_std"] = county_region_composite_stats["lcoe_std"].fillna(0)

def minmax_norm(s):
    denom = s.max() - s.min()
    if denom == 0:
        return pd.Series(0.0, index=s.index)
    return (s - s.min()) / denom

county_region_composite_stats["cf_std_norm"] = minmax_norm(county_region_composite_stats["cf_std"])
county_region_composite_stats["lcoe_std_norm"] = minmax_norm(county_region_composite_stats["lcoe_std"])

county_region_composite_stats["composite_variability_score"] = (
    W_CF * county_region_composite_stats["cf_std_norm"]
    + W_LCOE * county_region_composite_stats["lcoe_std_norm"]
)

county_region_composite_stats["capacity_weighted_score"] = (
    county_region_composite_stats["composite_variability_score"]
    * county_region_composite_stats["total_capacity_mw"]
)

display(
    county_region_composite_stats
    .sort_values("capacity_weighted_score", ascending=False)
    .head(20)
)

print("County-region groups:", len(county_region_composite_stats))
print("Baseline one per group:", len(county_region_composite_stats))
print("Remaining clusters to allocate:", TOTAL_POOL - len(county_region_composite_stats))

,model_region,county_group,candidate_rows,total_capacity_mw,cf_std,cf_min,cf_max,lcoe_std,lcoe_min,lcoe_max,cf_std_norm,lcoe_std_norm,composite_variability_score,capacity_weighted_score
27,AZ3,04001,1638,204076.885103,0.004173,0.296853,0.315813,1.949374,23.177225,33.471355,0.361489,0.237622,0.299556,61132.363164
428,NV2,32023,1115,115689.954867,0.006085,0.303274,0.329443,3.119859,26.146338,40.029358,0.527107,0.380301,0.453704,52488.994999
686,WY1,56037,1608,201334.765916,0.003155,0.283361,0.300811,1.834334,25.057420,34.986732,0.273318,0.223599,0.248459,50023.393003
211,ID1,16073,966,111092.781111,0.003766,0.275105,0.293689,3.642798,27.590698,45.306656,0.326246,0.444045,0.385146,42786.893049
30,AZ3,04017,1411,168893.042431,0.003761,0.299446,0.316826,1.436541,26.538240,33.393154,0.325831,0.175110,0.250470,42302.692207
426,NV2,32017,1067,115192.840058,0.004346,0.301417,0.322612,2.409459,27.909573,39.357310,0.376507,0.293705,0.335106,38601.836835
213,ID1,32007,1464,131287.102519,0.003824,0.281558,0.305815,1.756903,31.445559,42.291660,0.331233,0.214161,0.272697,35801.587930
0,AZ1,04005,1214,143152.699444,0.003467,0.300329,0.317243,1.394478,26.963272,35.567307,0.300307,0.169982,0.235145,33661.609337
405,NV1,32023,726,73454.564316,0.005050,0.296283,0.327574,3.124572,23.362118,41.945328,0.437477,0.380875,0.409176,30055.840331
497,OR3,41045,779,83072.528069,0.004979,0.273930,0.294699,2.241654,26.806355,35.714596,0.431317,0.273250,0.352284,29265.097634


County-region groups: 739
Baseline one per group: 739
Remaining clusters to allocate: 261


In [187]:
def allocate_by_score_with_caps(df, total_pool=1000, score_col="capacity_weighted_score"):
    out = df.copy()

    if total_pool < len(out):
        raise ValueError(
            f"total_pool={total_pool} is smaller than number of groups={len(out)}. "
            "Cannot give every group at least 1 cluster."
        )

    out["assigned_clusters_composite"] = 1
    out["max_possible_clusters"] = out["candidate_rows"].astype(int)
    out["remaining_capacity"] = (
        out["max_possible_clusters"] - out["assigned_clusters_composite"]
    ).clip(lower=0)

    remaining_budget = total_pool - out["assigned_clusters_composite"].sum()

    while remaining_budget > 0:
        eligible = out["remaining_capacity"] > 0

        if not eligible.any():
            print("No remaining candidate capacity. Could not use full budget.")
            break

        weights = out.loc[eligible, score_col].astype(float).clip(lower=0)

        if weights.sum() == 0:
            weights = out.loc[eligible, "total_capacity_mw"].astype(float).clip(lower=0)

        if weights.sum() == 0:
            weights = pd.Series(1.0, index=weights.index)

        raw_add = remaining_budget * weights / weights.sum()
        add_floor = np.floor(raw_add).astype(int)

        add_floor = np.minimum(
            add_floor,
            out.loc[eligible, "remaining_capacity"].astype(int),
        )

        if add_floor.sum() > 0:
            out.loc[eligible, "assigned_clusters_composite"] += add_floor
            out.loc[eligible, "remaining_capacity"] -= add_floor
            remaining_budget -= int(add_floor.sum())
        else:
            remainders = (raw_add - np.floor(raw_add)).sort_values(ascending=False)

            made_progress = False
            for idx in remainders.index:
                if remaining_budget <= 0:
                    break
                if out.loc[idx, "remaining_capacity"] > 0:
                    out.loc[idx, "assigned_clusters_composite"] += 1
                    out.loc[idx, "remaining_capacity"] -= 1
                    remaining_budget -= 1
                    made_progress = True

            if not made_progress:
                break

    return out

county_region_composite_alloc = allocate_by_score_with_caps(
    county_region_composite_stats,
    total_pool=TOTAL_POOL,
    score_col="capacity_weighted_score",
)

print("===== COMPOSITE COUNTY-REGION ALLOCATION =====")
print("Groups:", len(county_region_composite_alloc))
print("Assigned total:", county_region_composite_alloc["assigned_clusters_composite"].sum())
print("Groups with at least 1:", (county_region_composite_alloc["assigned_clusters_composite"] >= 1).sum())
print("Groups with more than 1:", (county_region_composite_alloc["assigned_clusters_composite"] > 1).sum())
print("Max assigned:", county_region_composite_alloc["assigned_clusters_composite"].max())

display(
    county_region_composite_alloc
    .sort_values("assigned_clusters_composite", ascending=False)
    .head(30)
)

===== COMPOSITE COUNTY-REGION ALLOCATION =====
Groups: 739
Assigned total: 1000
Groups with at least 1: 739
Groups with more than 1: 68
Max assigned: 31


,model_region,county_group,candidate_rows,total_capacity_mw,cf_std,cf_min,cf_max,lcoe_std,lcoe_min,lcoe_max,cf_std_norm,lcoe_std_norm,composite_variability_score,capacity_weighted_score,assigned_clusters_composite,max_possible_clusters,remaining_capacity
27,AZ3,04001,1638,204076.885103,0.004173,0.296853,0.315813,1.949374,23.177225,33.471355,0.361489,0.237622,0.299556,61132.363164,31,1638,1607
428,NV2,32023,1115,115689.954867,0.006085,0.303274,0.329443,3.119859,26.146338,40.029358,0.527107,0.380301,0.453704,52488.994999,22,1115,1093
686,WY1,56037,1608,201334.765916,0.003155,0.283361,0.300811,1.834334,25.057420,34.986732,0.273318,0.223599,0.248459,50023.393003,18,1608,1590
30,AZ3,04017,1411,168893.042431,0.003761,0.299446,0.316826,1.436541,26.538240,33.393154,0.325831,0.175110,0.250470,42302.692207,14,1411,1397
211,ID1,16073,966,111092.781111,0.003766,0.275105,0.293689,3.642798,27.590698,45.306656,0.326246,0.444045,0.385146,42786.893049,14,966,952
426,NV2,32017,1067,115192.840058,0.004346,0.301417,0.322612,2.409459,27.909573,39.357310,0.376507,0.293705,0.335106,38601.836835,12,1067,1055
213,ID1,32007,1464,131287.102519,0.003824,0.281558,0.305815,1.756903,31.445559,42.291660,0.331233,0.214161,0.272697,35801.587930,11,1464,1453
0,AZ1,04005,1214,143152.699444,0.003467,0.300329,0.317243,1.394478,26.963272,35.567307,0.300307,0.169982,0.235145,33661.609337,11,1214,1203
405,NV1,32023,726,73454.564316,0.005050,0.296283,0.327574,3.124572,23.362118,41.945328,0.437477,0.380875,0.409176,30055.840331,8,726,718
685,WY1,56035,522,59490.380793,0.003215,0.288020,0.304553,5.342305,28.571276,46.608837,0.278542,0.651210,0.464876,27655.632812,7,522,515


In [188]:
# Check which county_group is missing a county name in the composite output

tmp_check = county_region_composite_alloc.merge(
    county_lookup,
    on="county_group",
    how="left"
)

missing_counties = (
    tmp_check[tmp_check["county_name_full"].isna()][["county_group"]]
    .drop_duplicates()
)

print("Missing county lookup values:")
display(missing_counties)

Missing county lookup values:


,county_group
516,46102


In [189]:
# Manual county-name patch for county FIPS 46102
# This is only a traceability/display fix, not a modeling change.

manual_county_lookup = pd.DataFrame([
    {
        "county_group": "46102",
        "county_name": "Oglala Lakota County",
        "county_name_full": "Oglala Lakota County, SD",
        "state_abbrev": "SD",
    }
])

county_lookup = pd.concat(
    [county_lookup, manual_county_lookup],
    ignore_index=True
).drop_duplicates(subset=["county_group"], keep="last")

print(
    county_lookup[county_lookup["county_group"].eq("46102")]
)

     county_group          county_name_full           county_name state_abbrev
3235        46102  Oglala Lakota County, SD  Oglala Lakota County           SD


In [190]:
# Add county names to composite allocation output, then save

county_region_composite_alloc = county_region_composite_alloc.drop(
    columns=["county_name", "county_name_full", "state_abbrev"],
    errors="ignore"
).merge(
    county_lookup,
    on="county_group",
    how="left"
)

composite_out = OUT_DIR / "solar_cluster_allocation_county_region_1000_composite_cf_lcoe_capacity.csv"

county_region_composite_alloc.to_csv(composite_out, index=False)

print("Wrote:", composite_out)
print("Rows:", len(county_region_composite_alloc))
print("Rows missing county name:", county_region_composite_alloc["county_name_full"].isna().sum())

display(county_region_composite_alloc.head())

Wrote: /Users/laurenvo/Documents/Github/solar-county-analysis/clustering_county/outputs/allocation_1000/solar_cluster_allocation_county_region_1000_composite_cf_lcoe_capacity.csv
Rows: 739
Rows missing county name: 0


,model_region,county_group,candidate_rows,total_capacity_mw,cf_std,cf_min,cf_max,lcoe_std,lcoe_min,lcoe_max,cf_std_norm,lcoe_std_norm,composite_variability_score,capacity_weighted_score,assigned_clusters_composite,max_possible_clusters,remaining_capacity,county_name_full,county_name,state_abbrev
0,AZ1,04005,1214,143152.699444,0.003467,0.300329,0.317243,1.394478,26.963272,35.567307,0.300307,0.169982,0.235145,33661.609337,11,1214,1203,"Coconino County, AZ",NaN,NaN
1,AZ1,04012,377,36432.433853,0.002325,0.304883,0.314836,0.928475,23.984173,31.331976,0.201403,0.113178,0.157291,5730.479242,1,377,376,"La Paz County, AZ",NaN,NaN
2,AZ1,04013,51,4170.152205,0.002106,0.306351,0.312555,1.094344,27.556498,31.534460,0.182447,0.133397,0.157922,658.558737,1,51,50,"Maricopa County, AZ",NaN,NaN
3,AZ1,04015,981,87845.157974,0.003306,0.304315,0.322462,1.456630,22.663147,29.629540,0.286420,0.177558,0.231989,20379.138752,5,981,976,"Mohave County, AZ",NaN,NaN
4,AZ1,04025,497,46211.229409,0.003170,0.306299,0.320738,1.359513,25.640427,31.274826,0.274619,0.165720,0.220170,10174.317967,2,497,495,"Yavapai County, AZ",NaN,NaN


## Step 5D: County-only composite CF + LCOE + capacity-weighted allocation

In [191]:
# County-only version of the composite allocation.
# 419 counties get 1 baseline cluster each, then the remaining clusters are distributed
# using composite CF/LCOE variability weighted by total county capacity_mw.

TOTAL_POOL = 1000
W_CF = 0.5
W_LCOE = 0.5

county_composite_stats = (
    wecc.groupby([county_col])
    .agg(
        candidate_rows=(lcoe_col, "size"),
        total_capacity_mw=(cap_col, "sum"),
        cf_std=(cf_col, "std"),
        cf_min=(cf_col, "min"),
        cf_max=(cf_col, "max"),
        lcoe_std=(lcoe_col, "std"),
        lcoe_min=(lcoe_col, "min"),
        lcoe_max=(lcoe_col, "max"),
    )
    .reset_index()
)

county_composite_stats["cf_std"] = county_composite_stats["cf_std"].fillna(0)
county_composite_stats["lcoe_std"] = county_composite_stats["lcoe_std"].fillna(0)

county_composite_stats["cf_std_norm"] = minmax_norm(county_composite_stats["cf_std"])
county_composite_stats["lcoe_std_norm"] = minmax_norm(county_composite_stats["lcoe_std"])

county_composite_stats["composite_variability_score"] = (
    W_CF * county_composite_stats["cf_std_norm"]
    + W_LCOE * county_composite_stats["lcoe_std_norm"]
)

county_composite_stats["capacity_weighted_score"] = (
    county_composite_stats["composite_variability_score"]
    * county_composite_stats["total_capacity_mw"]
)

# Add county names
county_composite_stats = county_composite_stats.merge(
    county_lookup,
    left_on=county_col,
    right_on="county_group",
    how="left",
)

display(
    county_composite_stats
    .sort_values("capacity_weighted_score", ascending=False)
    .head(20)
)

print("County-only groups:", len(county_composite_stats))
print("Baseline one per county:", len(county_composite_stats))
print("Remaining clusters to allocate:", TOTAL_POOL - len(county_composite_stats))
print("Rows missing county name:", county_composite_stats["county_name_full"].isna().sum())

,county_group,candidate_rows,total_capacity_mw,cf_std,cf_min,cf_max,lcoe_std,lcoe_min,lcoe_max,cf_std_norm,lcoe_std_norm,composite_variability_score,capacity_weighted_score,county_name_full,county_name,state_abbrev
239,32023,1841,189144.519183,0.006673,0.296283,0.329443,3.891605,23.362118,41.945328,0.578106,0.451299,0.514703,97353.165853,"Nye County, NV",NaN,NaN
2,04005,2078,239935.439366,0.004185,0.293749,0.317243,3.062578,23.610842,42.467392,0.362509,0.355159,0.358834,86096.970194,"Coconino County, AZ",NaN,NaN
292,41045,1403,155530.335170,0.005488,0.273930,0.302586,4.374608,26.806355,44.730553,0.475451,0.507312,0.491381,76424.727186,"Malheur County, OR",NaN,NaN
0,04001,1720,211973.675664,0.004098,0.296853,0.315813,2.223027,23.177225,33.920410,0.355027,0.257799,0.306413,64951.449067,"Apache County, AZ",NaN,NaN
231,32007,2132,199916.985382,0.004333,0.281558,0.309496,2.009973,31.445559,43.308426,0.375399,0.233091,0.304245,60823.721827,"Elko County, NV",NaN,NaN
414,56037,1647,206710.038692,0.003152,0.283361,0.300811,2.656980,25.057420,42.561260,0.273060,0.308123,0.290591,60068.136516,"Sweetwater County, WY",NaN,NaN
8,04015,1447,133885.897559,0.004325,0.300358,0.322462,4.466206,22.663147,39.776836,0.374692,0.517934,0.446313,59755.028361,"Mohave County, AZ",NaN,NaN
282,41025,1279,144617.168678,0.004047,0.282265,0.306178,2.997894,30.727543,43.814934,0.350578,0.347658,0.349118,50488.427591,"Harney County, OR",NaN,NaN
9,04017,1556,179979.247346,0.003603,0.299446,0.316826,1.771087,26.538240,36.259624,0.312093,0.205388,0.258740,46567.898405,"Navajo County, AZ",NaN,NaN
168,16073,1091,126087.650664,0.003823,0.275105,0.295898,3.487391,27.590698,45.306656,0.331138,0.404423,0.367781,46372.628155,"Owyhee County, ID",NaN,NaN


County-only groups: 419
Baseline one per county: 419
Remaining clusters to allocate: 581
Rows missing county name: 0


In [192]:
# Allocate 1,000 clusters at the county-only level

county_composite_alloc = allocate_by_score_with_caps(
    county_composite_stats,
    total_pool=TOTAL_POOL,
    score_col="capacity_weighted_score",
)

# Rename the assigned column so it is clearly county-only
county_composite_alloc = county_composite_alloc.rename(
    columns={"assigned_clusters_composite": "assigned_clusters_county_composite"}
)

print("===== COMPOSITE COUNTY-ONLY ALLOCATION =====")
print("Counties:", len(county_composite_alloc))
print("Assigned total:", county_composite_alloc["assigned_clusters_county_composite"].sum())
print("Counties with at least 1:", (county_composite_alloc["assigned_clusters_county_composite"] >= 1).sum())
print("Counties with more than 1:", (county_composite_alloc["assigned_clusters_county_composite"] > 1).sum())
print("Max assigned:", county_composite_alloc["assigned_clusters_county_composite"].max())

display(
    county_composite_alloc
    .sort_values("assigned_clusters_county_composite", ascending=False)
    .head(30)
)

===== COMPOSITE COUNTY-ONLY ALLOCATION =====
Counties: 419
Assigned total: 1000
Counties with at least 1: 419
Counties with more than 1: 155
Max assigned: 42


,county_group,candidate_rows,total_capacity_mw,cf_std,cf_min,cf_max,lcoe_std,lcoe_min,lcoe_max,cf_std_norm,lcoe_std_norm,composite_variability_score,capacity_weighted_score,county_name_full,county_name,state_abbrev,assigned_clusters_county_composite,max_possible_clusters,remaining_capacity
239,32023,1841,189144.519183,0.006673,0.296283,0.329443,3.891605,23.362118,41.945328,0.578106,0.451299,0.514703,97353.165853,"Nye County, NV",NaN,NaN,42,1841,1799
2,04005,2078,239935.439366,0.004185,0.293749,0.317243,3.062578,23.610842,42.467392,0.362509,0.355159,0.358834,86096.970194,"Coconino County, AZ",NaN,NaN,33,2078,2045
292,41045,1403,155530.335170,0.005488,0.273930,0.302586,4.374608,26.806355,44.730553,0.475451,0.507312,0.491381,76424.727186,"Malheur County, OR",NaN,NaN,29,1403,1374
0,04001,1720,211973.675664,0.004098,0.296853,0.315813,2.223027,23.177225,33.920410,0.355027,0.257799,0.306413,64951.449067,"Apache County, AZ",NaN,NaN,22,1720,1698
8,04015,1447,133885.897559,0.004325,0.300358,0.322462,4.466206,22.663147,39.776836,0.374692,0.517934,0.446313,59755.028361,"Mohave County, AZ",NaN,NaN,21,1447,1426
414,56037,1647,206710.038692,0.003152,0.283361,0.300811,2.656980,25.057420,42.561260,0.273060,0.308123,0.290591,60068.136516,"Sweetwater County, WY",NaN,NaN,21,1647,1626
231,32007,2132,199916.985382,0.004333,0.281558,0.309496,2.009973,31.445559,43.308426,0.375399,0.233091,0.304245,60823.721827,"Elko County, NV",NaN,NaN,21,2132,2111
282,41025,1279,144617.168678,0.004047,0.282265,0.306178,2.997894,30.727543,43.814934,0.350578,0.347658,0.349118,50488.427591,"Harney County, OR",NaN,NaN,16,1279,1263
168,16073,1091,126087.650664,0.003823,0.275105,0.295898,3.487391,27.590698,45.306656,0.331138,0.404423,0.367781,46372.628155,"Owyhee County, ID",NaN,NaN,15,1091,1076
9,04017,1556,179979.247346,0.003603,0.299446,0.316826,1.771087,26.538240,36.259624,0.312093,0.205388,0.258740,46567.898405,"Navajo County, AZ",NaN,NaN,15,1556,1541


In [193]:
# Optional: split the county-only allocation back to county-region pairs using capacity share.
# This helps compare the county-only idea to the current PowerGenome county-region setup.

county_region_capacity_share = (
    wecc.groupby([county_col, "model_region"])
    .agg(
        county_region_candidate_rows=(lcoe_col, "size"),
        county_region_capacity_mw=(cap_col, "sum"),
    )
    .reset_index()
)

county_capacity_totals = (
    county_region_capacity_share.groupby(county_col)
    .agg(
        county_total_capacity_mw=("county_region_capacity_mw", "sum"),
        n_model_regions=("model_region", "nunique"),
    )
    .reset_index()
)

county_region_capacity_share = county_region_capacity_share.merge(
    county_capacity_totals,
    on=county_col,
    how="left",
)

county_region_capacity_share["capacity_share_of_county"] = (
    county_region_capacity_share["county_region_capacity_mw"]
    / county_region_capacity_share["county_total_capacity_mw"]
)

county_only_split_to_regions = county_region_capacity_share.merge(
    county_composite_alloc[
        [
            county_col,
            "county_name_full",
            "assigned_clusters_county_composite",
        ]
    ],
    on=county_col,
    how="left",
)

county_only_split_to_regions["raw_region_cluster_share"] = (
    county_only_split_to_regions["assigned_clusters_county_composite"]
    * county_only_split_to_regions["capacity_share_of_county"]
)

display(
    county_only_split_to_regions
    .sort_values(["assigned_clusters_county_composite", county_col], ascending=[False, True])
    .head(30)
)

print("County-region rows after splitting county-only allocation:", len(county_only_split_to_regions))
print("Counties represented:", county_only_split_to_regions[county_col].nunique())
print("Total county-level assigned clusters:", county_composite_alloc["assigned_clusters_county_composite"].sum())

,county_group,model_region,county_region_candidate_rows,county_region_capacity_mw,county_total_capacity_mw,n_model_regions,capacity_share_of_county,county_name_full,assigned_clusters_county_composite,raw_region_cluster_share
400,32023,NV1,726,73454.564316,189144.519183,2,0.388352,"Nye County, NV",42,16.310764
401,32023,NV2,1115,115689.954867,189144.519183,2,0.611648,"Nye County, NV",42,25.689236
4,04005,AZ1,1214,143152.699444,239935.439366,5,0.596630,"Coconino County, AZ",33,19.688793
5,04005,AZ2,225,17330.192205,239935.439366,5,0.072229,"Coconino County, AZ",33,2.383543
6,04005,AZ3,526,67589.597774,239935.439366,5,0.281699,"Coconino County, AZ",33,9.296070
7,04005,AZ4,1,10.125000,239935.439366,5,0.000042,"Coconino County, AZ",33,0.001393
8,04005,NV2,112,11852.824943,239935.439366,5,0.049400,"Coconino County, AZ",33,1.630202
494,41045,ID1,424,48482.893119,155530.335170,3,0.311726,"Malheur County, OR",29,9.040062
495,41045,NV1,200,23974.913982,155530.335170,3,0.154149,"Malheur County, OR",29,4.470334
496,41045,OR3,779,83072.528069,155530.335170,3,0.534124,"Malheur County, OR",29,15.489604


County-region rows after splitting county-only allocation: 739
Counties represented: 419
Total county-level assigned clusters: 1000


In [194]:
# Save county-only composite outputs

county_composite_out = OUT_DIR / "solar_cluster_allocation_county_only_1000_composite_cf_lcoe_capacity.csv"
county_only_split_out = OUT_DIR / "solar_cluster_allocation_county_only_1000_composite_split_to_regions.csv"

county_composite_alloc.to_csv(county_composite_out, index=False)
county_only_split_to_regions.to_csv(county_only_split_out, index=False)

print("Wrote:", county_composite_out)
print("Wrote:", county_only_split_out)

Wrote: /Users/laurenvo/Documents/Github/solar-county-analysis/clustering_county/outputs/allocation_1000/solar_cluster_allocation_county_only_1000_composite_cf_lcoe_capacity.csv
Wrote: /Users/laurenvo/Documents/Github/solar-county-analysis/clustering_county/outputs/allocation_1000/solar_cluster_allocation_county_only_1000_composite_split_to_regions.csv


## Step 5E: Integer county-only allocation split back to county-region pairs

In [195]:
# Convert the fractional county-only split-back table into integer county-region cluster counts.
#
# Goal:
# - Preserve the county-only allocation exactly.
# - For each county, assigned county-region clusters sum back to the county's assigned cluster count.
# - Allocate across model regions using the largest-remainder method based on capacity share.
# - Cap each county-region allocation by its available candidate rows.

import numpy as np

def apportion_county_clusters_to_regions(
    g,
    county_allocation_col="assigned_clusters_county_composite",
    raw_share_col="raw_region_cluster_share",
    region_cap_col="county_region_candidate_rows",
):
    g = g.copy()

    target = int(round(g[county_allocation_col].iloc[0]))
    total_region_cap = int(g[region_cap_col].sum())

    # Safety cap: should usually not bind because the county-level allocation
    # was already capped by total county candidate rows.
    target = min(target, total_region_cap)

    # Start with floor of the fractional allocation
    g["assigned_clusters_county_to_region_integer"] = np.floor(
        g[raw_share_col]
    ).astype(int)

    # Cap by candidate rows in each county-region pair
    g["assigned_clusters_county_to_region_integer"] = np.minimum(
        g["assigned_clusters_county_to_region_integer"],
        g[region_cap_col].astype(int),
    )

    g["fractional_remainder"] = (
        g[raw_share_col]
        - np.floor(g[raw_share_col])
    )

    current_total = int(g["assigned_clusters_county_to_region_integer"].sum())
    remaining = target - current_total

    # Distribute remaining clusters by largest remainder,
    # breaking ties using capacity share and available capacity.
    while remaining > 0:
        eligible = g[
            g["assigned_clusters_county_to_region_integer"]
            < g[region_cap_col].astype(int)
        ].copy()

        if eligible.empty:
            break

        eligible = eligible.sort_values(
            [
                "fractional_remainder",
                "capacity_share_of_county",
                "county_region_capacity_mw",
            ],
            ascending=[False, False, False],
        )

        made_progress = False

        for idx in eligible.index:
            if remaining <= 0:
                break

            if (
                g.loc[idx, "assigned_clusters_county_to_region_integer"]
                < int(g.loc[idx, region_cap_col])
            ):
                g.loc[idx, "assigned_clusters_county_to_region_integer"] += 1
                remaining -= 1
                made_progress = True

        if not made_progress:
            break

    return g


county_only_split_to_regions_integer = (
    county_only_split_to_regions
    .groupby(county_col, group_keys=False)
    .apply(apportion_county_clusters_to_regions)
    .reset_index(drop=True)
)

# QA: make sure integer split sums back to each county's assigned cluster count
integer_split_check = (
    county_only_split_to_regions_integer
    .groupby(county_col)
    .agg(
        assigned_clusters_county_composite=("assigned_clusters_county_composite", "first"),
        assigned_clusters_split_to_regions_integer=(
            "assigned_clusters_county_to_region_integer",
            "sum",
        ),
        n_model_regions=("model_region", "nunique"),
        positive_region_allocations=(
            "assigned_clusters_county_to_region_integer",
            lambda s: (s > 0).sum(),
        ),
    )
    .reset_index()
)

integer_split_check["matches_county_assignment"] = (
    integer_split_check["assigned_clusters_county_composite"]
    == integer_split_check["assigned_clusters_split_to_regions_integer"]
)

print("===== INTEGER COUNTY-ONLY SPLIT BACK TO COUNTY-REGION PAIRS =====")
print("County-region rows:", len(county_only_split_to_regions_integer))
print("Counties represented:", county_only_split_to_regions_integer[county_col].nunique())
print(
    "Total integer assigned clusters:",
    county_only_split_to_regions_integer[
        "assigned_clusters_county_to_region_integer"
    ].sum(),
)
print(
    "County-region pairs with >0 clusters:",
    (
        county_only_split_to_regions_integer[
            "assigned_clusters_county_to_region_integer"
        ] > 0
    ).sum(),
)
print(
    "County-region pairs with 0 clusters:",
    (
        county_only_split_to_regions_integer[
            "assigned_clusters_county_to_region_integer"
        ] == 0
    ).sum(),
)
print(
    "Counties where integer split matches county assignment:",
    integer_split_check["matches_county_assignment"].sum(),
    "/",
    len(integer_split_check),
)

mismatches = integer_split_check[
    ~integer_split_check["matches_county_assignment"]
]

print("Mismatched counties:", len(mismatches))

display(
    county_only_split_to_regions_integer
    .sort_values(
        [
            "assigned_clusters_county_composite",
            "assigned_clusters_county_to_region_integer",
            county_col,
        ],
        ascending=[False, False, True],
    )
    .head(30)
)

if len(mismatches) > 0:
    display(mismatches)

===== INTEGER COUNTY-ONLY SPLIT BACK TO COUNTY-REGION PAIRS =====
County-region rows: 739
Counties represented: 419
Total integer assigned clusters: 1000
County-region pairs with >0 clusters: 489
County-region pairs with 0 clusters: 250
Counties where integer split matches county assignment: 419 / 419
Mismatched counties: 0


,county_group,model_region,county_region_candidate_rows,county_region_capacity_mw,county_total_capacity_mw,n_model_regions,capacity_share_of_county,county_name_full,assigned_clusters_county_composite,raw_region_cluster_share,assigned_clusters_county_to_region_integer,fractional_remainder
401,32023,NV2,1115,115689.954867,189144.519183,2,0.611648,"Nye County, NV",42,25.689236,26,0.689236
400,32023,NV1,726,73454.564316,189144.519183,2,0.388352,"Nye County, NV",42,16.310764,16,0.310764
4,04005,AZ1,1214,143152.699444,239935.439366,5,0.596630,"Coconino County, AZ",33,19.688793,20,0.688793
6,04005,AZ3,526,67589.597774,239935.439366,5,0.281699,"Coconino County, AZ",33,9.296070,9,0.296070
5,04005,AZ2,225,17330.192205,239935.439366,5,0.072229,"Coconino County, AZ",33,2.383543,2,0.383543
8,04005,NV2,112,11852.824943,239935.439366,5,0.049400,"Coconino County, AZ",33,1.630202,2,0.630202
7,04005,AZ4,1,10.125000,239935.439366,5,0.000042,"Coconino County, AZ",33,0.001393,0,0.001393
496,41045,OR3,779,83072.528069,155530.335170,3,0.534124,"Malheur County, OR",29,15.489604,16,0.489604
494,41045,ID1,424,48482.893119,155530.335170,3,0.311726,"Malheur County, OR",29,9.040062,9,0.040062
495,41045,NV1,200,23974.913982,155530.335170,3,0.154149,"Malheur County, OR",29,4.470334,4,0.470334


In [196]:
county_only_integer_split_out = OUT_DIR / "solar_cluster_allocation_county_only_1000_composite_integer_split_to_regions.csv"

county_only_split_to_regions_integer.to_csv(county_only_integer_split_out, index=False)

print("Wrote:", county_only_integer_split_out)

Wrote: /Users/laurenvo/Documents/Github/solar-county-analysis/clustering_county/outputs/allocation_1000/solar_cluster_allocation_county_only_1000_composite_integer_split_to_regions.csv


## Step 6: Save allocation outputs

This section saves the allocation results to CSV files.

The notebook writes three main outputs:

1. `solar_cluster_allocation_county_region_1000.csv`  
   Allocation table using `(model_region, county_group)` groups.

2. `solar_cluster_allocation_county_only_1000.csv`  
   Allocation table using unique counties only.

3. `solar_cluster_allocation_1000_summary.txt`  
   Short text summary of the allocation results.

The county-region CSV is the more direct input for a future PowerGenome implementation. The county-only CSV is useful for reviewing the conceptual allocation method at the county level.

In [ ]:
summary = f"""
Input file:
{SOLAR_META_PATH}

Old metric-only method:
metric used = {metric_col}

County-region LCOE-only allocation:
groups = {len(county_region_alloc)}
assigned_clusters_total = {county_region_alloc["assigned_clusters"].sum()}
groups_with_at_least_1 = {(county_region_alloc["assigned_clusters"] >= 1).sum()}
groups_with_more_than_1 = {(county_region_alloc["assigned_clusters"] > 1).sum()}
max_assigned_clusters = {county_region_alloc["assigned_clusters"].max()}

County-only LCOE-only allocation:
counties = {len(county_alloc)}
assigned_clusters_total = {county_alloc["assigned_clusters"].sum()}
counties_with_at_least_1 = {(county_alloc["assigned_clusters"] >= 1).sum()}
counties_with_more_than_1 = {(county_alloc["assigned_clusters"] > 1).sum()}
max_assigned_clusters = {county_alloc["assigned_clusters"].max()}

Composite CF + LCOE + capacity-weighted county-region allocation:
groups = {len(county_region_composite_alloc)}
assigned_clusters_total = {county_region_composite_alloc["assigned_clusters_composite"].sum()}
groups_with_at_least_1 = {(county_region_composite_alloc["assigned_clusters_composite"] >= 1).sum()}
groups_with_more_than_1 = {(county_region_composite_alloc["assigned_clusters_composite"] > 1).sum()}
max_assigned_clusters = {county_region_composite_alloc["assigned_clusters_composite"].max()}

Composite method:
- Each county-region group gets 1 baseline cluster.
- Remaining clusters are allocated using:
  0.5 * normalized CF std + 0.5 * normalized LCOE std
- That composite variability score is multiplied by total available capacity_mw.
- Assigned clusters are capped by the number of candidate solar rows available.

County-only composite CF + LCOE + capacity-weighted allocation:
counties = {len(county_composite_alloc)}
assigned_clusters_total = {county_composite_alloc["assigned_clusters_county_composite"].sum()}
counties_with_at_least_1 = {(county_composite_alloc["assigned_clusters_county_composite"] >= 1).sum()}
counties_with_more_than_1 = {(county_composite_alloc["assigned_clusters_county_composite"] > 1).sum()}
max_assigned_clusters = {county_composite_alloc["assigned_clusters_county_composite"].max()}

County-only composite split back to county-region pairs:
county_region_rows = {len(county_only_split_to_regions)}
counties_represented = {county_only_split_to_regions[county_col].nunique()}

County-only composite integer split back to county-region pairs:
county_region_rows = {len(county_only_split_to_regions_integer)}
assigned_clusters_total = {county_only_split_to_regions_integer["assigned_clusters_county_to_region_integer"].sum()}
counties_represented = {county_only_split_to_regions_integer[county_col].nunique()}
county_region_pairs_with_positive_clusters = {(county_only_split_to_regions_integer["assigned_clusters_county_to_region_integer"] > 0).sum()}
county_region_pairs_with_zero_clusters = {(county_only_split_to_regions_integer["assigned_clusters_county_to_region_integer"] == 0).sum()}
county_assignment_mismatches = {len(integer_split_check[~integer_split_check["matches_county_assignment"]])}

Notes:
- County-region allocation matches the current PowerGenome WECC setup more directly.
- County-only allocation matches conceptual county-level example more directly.
- The composite county-region output is the main test for suggested CF + LCOE + capacity-weighted approach.
"""

print(summary)


Input file:
/Users/laurenvo/Documents/Switch-USA-PG-ReEDS/pg/extra_inputs/resource_groups_wecc_county_solar_test/ReEDS-cpas-patched/solar_lcoe_ReEDS_pg_schema_with_county_group.csv

Old metric-only method:
metric used = lcoe

County-region LCOE-only allocation:
groups = 739
assigned_clusters_total = 1000
groups_with_at_least_1 = 739
groups_with_more_than_1 = 133
max_assigned_clusters = 24

County-only LCOE-only allocation:
counties = 419
assigned_clusters_total = 1000
counties_with_at_least_1 = 419
counties_with_more_than_1 = 230
max_assigned_clusters = 63

Composite CF + LCOE + capacity-weighted county-region allocation:
groups = 739
assigned_clusters_total = 1000
groups_with_at_least_1 = 739
groups_with_more_than_1 = 68
max_assigned_clusters = 31

Composite method:
- Each county-region group gets 1 baseline cluster.
- Remaining clusters are allocated using:
  0.5 * normalized CF std + 0.5 * normalized LCOE std
- That composite variability score is multiplied by total available capac

: 

## Interpretation

This notebook creates a pre-allocation plan for a fixed 1,000-cluster solar budget.

The main takeaway is that a single global PowerGenome setting like `n_clusters: 5` gives up to 5 clusters per group, but it does not target a specific total number of clusters. If we want a target like 1,000 total solar clusters, we need a separate allocation step like this.

The county-region allocation is most consistent with the current PowerGenome WECC setup because PowerGenome is currently clustering by model region and county.

The county-only allocation is useful conceptually because it answers the question: "If every county gets one baseline cluster, where should the remaining clusters go?"

In both cases, the allocation:
- gives every valid group at least 1 cluster,
- distributes the remaining clusters using LCOE standard deviation,
- and avoids assigning more clusters than available candidate solar rows.